# Fine-mapping postprocessing

## Description

Postprocessing utilities for the S4 `QtlFineMappingResult` / `GwasFineMappingResult` RDSes produced by `fine_mapping.ipynb`, `multivariate_fine_mapping.ipynb`, `functional_fine_mapping.ipynb`, or `rss_analysis.ipynb`. Replaces the legacy `mnm_analysis/mnm_postprocessing.ipynb` (which consumed the older list-shaped RDS layout).

All steps consume FMR RDSes through the pecotmr S4 accessor surface (`getPip`, `getCs`, `getTopLoci`, `getMarginalEffects`, `getVariantIds`) so they work uniformly for QTL and GWAS outputs.

Workflow steps (all independent &mdash; pick what you need):

- **`[export_top_loci]`** &mdash; concatenate every input RDS's top-loci table into one TSV (one row per variant, with `study / context / trait / region_id / method / source` identifier columns).
- **`[export_credible_sets]`** &mdash; same shape, but the per-row CS membership.
- **`[export_pip]`** &mdash; raw `variant_id / pip` long table across all inputs (filterable with `--signal-cutoff`).
- **`[export_marginals]`** &mdash; per-variant marginal effects (`beta / se / z / p`) across all inputs.
- **`[pip_landscape_plot]`** &mdash; per-RDS PIP-vs-position scatter, one facet per FMR row.
- **`[upset_cs_overlap]`** &mdash; UpSet plot of credible-set variant overlap across one or more RDSes (one set per `(study, context, trait, method)` tuple for QTL; per `(study, method, region_id)` for GWAS).

## Inputs

- `--fmr-list <RDS> [<RDS> ...]` &mdash; one or more FineMappingResult RDS files (mix QTL and GWAS freely; identifier columns adapt per class).
- `--name` &mdash; output filename prefix.
- `--signal-cutoff` &mdash; PIP cutoff for `export_top_loci` / `export_pip`. Default 0 (no filter); pass 0.025 to mirror the legacy susie default.
- `--max-sets` &mdash; cap on UpSet set count (default 20).
- `--cwd` / `--modular-script-dir` &mdash; standard SoS infra.

## Outputs

Per-step:

- `{cwd}/{name}.topLoci.tsv.gz`
- `{cwd}/{name}.cs.tsv.gz`
- `{cwd}/{name}.pip.tsv.gz`
- `{cwd}/{name}.marginals.tsv.gz`
- `{cwd}/plots/{name}.{rds-basename}.pip_landscape.png` (one per input RDS)
- `{cwd}/plots/{name}.upset.png`


## Example

```bash
# Top-loci TSV + UpSet plot across every fine-mapping RDS in a run:
sos run pipeline/fine_mapping_postprocessing.ipynb \
    export_top_loci+upset_cs_overlap \
    --cwd output/postprocessing \
    --modular-script-dir /path/to/code/script \
    --name myrun \
    --signal-cutoff 0.025 \
    --fmr-list output/fine_mapping/*.finemap.rds

# All four TSV exports + per-RDS PIP plot:
sos run pipeline/fine_mapping_postprocessing.ipynb \
    export_top_loci+export_credible_sets+export_pip+export_marginals+pip_landscape_plot \
    --cwd output/postprocessing --name myrun \
    --fmr-list output/fine_mapping/*.finemap.rds
```


In [ ]:
[global]
parameter: cwd = path('output')
parameter: modular_script_dir = path('code/script')
parameter: name = str
# One or more FineMappingResult RDS paths (mix QTL + GWAS freely).
parameter: fmr_list = []
parameter: signal_cutoff = 0.0
parameter: max_sets = 20
parameter: container = ''
parameter: job_size = 1
parameter: walltime = '30m'
parameter: mem = '8G'
parameter: numThreads = 1



In [ ]:
[export_top_loci]
stop_if(not fmr_list, '--fmr-list requires at least one RDS path.')
paths = [str(p) for p in fmr_list]
input: paths
output: f"{cwd}/{name}.topLoci.tsv.gz"
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads, tags = f"{step_name}_{_output:bn}"
bash: expand = '${ }', stderr = f"{_output}.stderr", stdout = f"{_output}.stdout", container = container
    Rscript ${modular_script_dir}/pecotmr_integration/fine_mapping_export.R \
        --input ${_input} \
        --view topLoci \
        --signal-cutoff ${signal_cutoff} \
        --output ${_output}

In [ ]:
[export_credible_sets]
stop_if(not fmr_list, '--fmr-list requires at least one RDS path.')
paths = [str(p) for p in fmr_list]
input: paths
output: f"{cwd}/{name}.cs.tsv.gz"
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads, tags = f"{step_name}_{_output:bn}"
bash: expand = '${ }', stderr = f"{_output}.stderr", stdout = f"{_output}.stdout", container = container
    Rscript ${modular_script_dir}/pecotmr_integration/fine_mapping_export.R \
        --input ${_input} \
        --view cs \
        --output ${_output}

In [ ]:
[export_pip]
stop_if(not fmr_list, '--fmr-list requires at least one RDS path.')
paths = [str(p) for p in fmr_list]
input: paths
output: f"{cwd}/{name}.pip.tsv.gz"
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads, tags = f"{step_name}_{_output:bn}"
bash: expand = '${ }', stderr = f"{_output}.stderr", stdout = f"{_output}.stdout", container = container
    Rscript ${modular_script_dir}/pecotmr_integration/fine_mapping_export.R \
        --input ${_input} \
        --view pip \
        --signal-cutoff ${signal_cutoff} \
        --output ${_output}

In [ ]:
[export_marginals]
stop_if(not fmr_list, '--fmr-list requires at least one RDS path.')
paths = [str(p) for p in fmr_list]
input: paths
output: f"{cwd}/{name}.marginals.tsv.gz"
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads, tags = f"{step_name}_{_output:bn}"
bash: expand = '${ }', stderr = f"{_output}.stderr", stdout = f"{_output}.stdout", container = container
    Rscript ${modular_script_dir}/pecotmr_integration/fine_mapping_export.R \
        --input ${_input} \
        --view marginals \
        --output ${_output}

In [ ]:
[pip_landscape_plot]
stop_if(not fmr_list, '--fmr-list requires at least one RDS path.')
paths = [str(p) for p in fmr_list]
# One PNG per input RDS (per-row fan-out).
input: paths, group_by = 1
output: f"{cwd}/plots/{name}.{_input:bn}.pip_landscape.png"
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads, tags = f"{step_name}_{_output:bn}"
bash: expand = '${ }', stderr = f"{_output}.stderr", stdout = f"{_output}.stdout", container = container
    Rscript ${modular_script_dir}/pecotmr_integration/fine_mapping_pip_plot.R \
        --input ${_input} \
        --output ${_output}

In [ ]:
[upset_cs_overlap]
stop_if(not fmr_list, '--fmr-list requires at least one RDS path.')
paths = [str(p) for p in fmr_list]
input: paths
output: f"{cwd}/plots/{name}.upset.png"
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads, tags = f"{step_name}_{_output:bn}"
bash: expand = '${ }', stderr = f"{_output}.stderr", stdout = f"{_output}.stdout", container = container
    Rscript ${modular_script_dir}/pecotmr_integration/fine_mapping_upset.R \
        --input ${_input} \
        --max-sets ${max_sets} \
        --output ${_output}